# 电路中的电压和电流scikit-rf 的 `Circuit` 对象允许推算出电路中所有节点处的电压和电流，给定电路端口的功率（和相位）激励。以下是一些示例，用于阐明电流方向所使用的约定。

In [ ]:
%matplotlib inline

In [ ]:
# standard imports
import numpy as np

import skrf as rf
from skrf.circuit import Circuit

In [ ]:
rf.stylely()

## 一个简单的传输线首先，我们假设一个由信号源激励的简单传输线。我们假设信号源与传输线的阻抗匹配（$Z_s=Z_0$），并且传输线连接到一个匹配的负载（$Z_0=Z_L$），均为 50 欧姆。[图片：circuit_vi_simple_line.svg]对于给定的输入功率（和相位），这条传输线的输入端和输出端的射频电流和电压是多少？

In [ ]:
P_f = 1  # forward power in Watt
Z = 50  # source internal impedance, line characteristic impedance and load impedance
L = 10  # line length in [m]

freq = rf.Frequency(2, 2, 1, unit='GHz')
line_media = rf.media.DefinedGammaZ0(freq, z0=Z)  # lossless line medium
line = line_media.line(d=L, unit='m', name='line')  # transmission line Network

假设信号源产生一个功率为 $P_f$、相位为 $\phi$ 的输入信号，那么对于这样的传输线，在传输线入口处的电压和电流为：$$V_{in} = \sqrt{2 Z_0 P_f} e^{j \phi}$$$$I_{in} = \sqrt{2 \frac{P_f}{Z_0}} e^{j \phi}$$

In [ ]:
V_in = np.sqrt(2*Z*P_f)
I_in = np.sqrt(2*P_f/Z)
print(f'Input voltage and current: {V_in} V and {I_in} A')

电压和电流沿着传输线变化。可以使用 scikit-rf 提供的传输线工具来计算线路输出端的电压和电流：

In [ ]:
theta = rf.tlineFunctions.theta(line_media.gamma, freq.f, L)  # electrical length
V_out, I_out = rf.tlineFunctions.voltage_current_propagation(V_in, I_in, Z, theta)
print(f'Output voltage and current: {V_out} V and {I_out} A')

让我们使用 `Circuit` 执行相同的计算。首先，我们需要定义电路，即创建输入/输出端口，并将这些端口连接到我们已经创建的传输线网络。然后，我们可以构建电路：

In [ ]:
port1 = Circuit.Port(frequency=freq, name='port1', z0=50)
port2 = Circuit.Port(frequency=freq, name='port2', z0=50)
cnx = [
    [(port1, 0), (line, 0)],
    [(port2, 0), (line, 1)]
]
crt = Circuit(cnx)

检查电路的图是否符合预期始终是一个好的做法。在这种情况下，该图非常简单：两个端口连接到一个具有两个端口的网络。

In [ ]:
crt.plot_graph(network_labels=True, edge_labels=True, inter_labels=True)

`Circuit` 提供了两种方法来确定电路输入/输出端口（也称为“外部端口”）处的电压和电流。这两种方法都以每个端口的功率和相位作为输入：

In [ ]:
power = [1, 0]  # 1 Watt at port1 and 0 at port2
phase = [0, 0]  # 0 radians

In [ ]:
V_at_ports = crt.voltages_external(power, phase)
print(V_at_ports)

In [ ]:
I_at_ports = crt.currents_external(power, phase)
print(I_at_ports)

结果与之前类似，只是端口 2 处的电流方向相反。这是正常的，因为 `currents_external()` 方法将电流的正方向定义为“进入”电路的方向。关于此约定，更多细节请参见本示例的末尾。

## 具有不同特性阻抗的同轴 T 型结构为了提供一个更高级的示例，我们在全波软件（此处为 ANSYS HFSS，但其他软件也可以）中构建了以下结构：一个同轴 T 型结构，其定义中使用了不同的同轴横截面（因此具有不同的特性阻抗）。<img src="circuit_vi_HFSS_Coaxial_T.png" with="800">

所有三个端口都以不同的功率和相位进行激励，以增加一些复杂性，如下所示：| 端口 | 功率 [W] | 相位 [度] || --- | --- | --- || #1 | 1 | -10 || #2 | 2 | -20 || #3 | 3 | +60 |### 全波参考解电压和电流直接在全波软件中进行计算。电压是通过沿从端口横截面的内导体到外导体的直线进行积分来获得的。电流是通过沿包围端口横截面内导体的圆进行积分来获得的。请注意，这些圆的方向是为了定义正电流方向，即指向端口内部的方向（右手定则）。这里给出了在 3 个频率下的全波解，供参考：

In [ ]:
import pandas as pd  # convenient to read .csv files

pd.read_csv('circuit_vi_HFSS_Voltages.csv')

In [ ]:
pd.read_csv('circuit_vi_HFSS_Currents.csv')

### 在电路模拟器中电压和电流也可以使用电路模拟器来推导，例如：<img src="circuit_vi_Designer_circuit.png" width="800">（其中，电流探头的方向设置成当电流流经网络时，电流为正）。这会得到基本相同的结果：

In [ ]:
pd.read_csv('circuit_vi_Designer_Voltages.csv')

In [ ]:
pd.read_csv('circuit_vi_Designer_Currents.csv')

### 使用 `Circuit`现在让我们使用 scikit-rf 的 `Circuit` 构建相同的电路：

In [ ]:
# Importing the 3-port .s3p file exported from full-wave simulation
coaxial_T = rf.Network('circuit_vi_Coaxial_T.s3p')
# pay attention to the port's characteristic impedance
# it should match the Network characteristic impedances otherwise this will generate mismatches
port1 = Circuit.Port(coaxial_T.frequency, 'port1', coaxial_T.z0[:,0])
port2 = Circuit.Port(coaxial_T.frequency, 'port2', coaxial_T.z0[:,1])
port3 = Circuit.Port(coaxial_T.frequency, 'port3', coaxial_T.z0[:,2])
# connection list
cnx = [
    [(port1, 0), (coaxial_T, 0)],
    [(port2, 0), (coaxial_T, 1)],
    [(port3, 0), (coaxial_T, 2)]
]
# building the circuit
crt = Circuit(cnx)

In [ ]:
# let's check if our connection list is correctly defined
crt.plot_graph(network_labels=True, edge_labels=True, inter_labels=True)

对于给定的激励信号，端口处的电压和电流为：

In [ ]:
power = [1, 2, 3]  # input power in watts at ports 1, 2 and 3
phase = np.deg2rad([-10, -20, +60])  # input phase in rad at ports 1, 2 and 3

voltages = crt.voltages_external(power, phase)
currents = crt.currents_external(power, phase)

In [ ]:
# just for a better rendering in the notebook
pd.concat([
    pd.DataFrame(np.abs(voltages), columns=['mag V1', 'mag V2', 'mag V3'], index=crt.frequency.f/1e6),
    pd.DataFrame(np.angle(voltages, deg=True), columns=['Phase V1', 'Phase V2', 'Phase V3'], index=crt.frequency.f/1e6)
], axis=1)

In [ ]:
# just for a better rendering in the notebook
pd.concat([
    pd.DataFrame(np.abs(currents), columns=['mag I1', 'mag I2', 'mag I3'], index=crt.frequency.f/1e6),
    pd.DataFrame(np.angle(currents, deg=True), columns=['Phase I1', 'Phase I2', 'Phase I3'], index=crt.frequency.f/1e6)
], axis=1)

这些结果与全波计算的结果非常吻合，太好了。

## 外部端口与内部端口？您可能在 `Circuit` 文档中注意到，我们经常讨论“内部”或“内部端口”以及“外部”或“外部端口”。外部端口对应于在构建 `Circuit` 时定义的 `Circuit.Port()` 网络。内部端口是 `Circuit` 内部的所有其他连接。`Circuit` 算法允许访问电路内部的电压和电流。在前面的示例中，内部端口不多，因为我们直接将一个 N 端口连接到外部端口。但是，在更复杂的应用中，我们可以有很多内部端口（请参阅其他 `Circuit` 示例）。在 `Circuit` 中，电压和电流是峰值。虽然电压的定义是明确的，但正电流可以以一种或另一种方式定义，从而导致 180 度的歧义。为了解决这个问题，我们选择了以下定义：内部电流的定义方式是，当电流流向网络时，测量值为正。因此，您会发现，“外部”电流与相应端口处的“内部”电流的符号相反，因为内部电流实际上是流向端口网络。

In [ ]:
# internals currents (currents at all connexions)
# in this example, there are 6 internal connexions (3 pairs of connexions)
crt.currents(power, phase)

In [ ]:
# This gives the indices of the "external" ports
crt.port_indexes

In [ ]:
# So we can keep only "external" ports
crt.currents(power, phase)[:, crt.port_indexes]

In [ ]:
# note the sign difference due to the convention chosen for internal ports
crt.currents_external(power, phase)

下图使用之前的示例来说明这些差异：<img src="circuit_vi_convention_currents.png" width="800">

## 复杂电路中的电压和电流### 功率波和行波在 `scikit-rf` 中，电压和电流的计算依赖于电路的散射参数（S 参数）。S 参数的默认定义使用功率波，表示如下：$$ s_{ij} = \frac{b_i}{a_j} $$其中，$b_i$ 和 $a_j$ 分别表示给定网络中端口 `i` 和 `j` 处的输出和输入功率波。例如，方法 `Circuit._b()` 和 `Circuit._a()` 提供了计算这些输出和输入功率波的支持，尽管对于典型的应用场景，这通常是不必要的。

In [ ]:
# To obtain the input/output power waves under given phase and amplitude, you can use:
a = crt._a(crt._a_external(power, phase))
b = crt._b(a)

根据 `S-parameters` 的定义，我们可以确定在给定的 `Network` 中端口 `i` 和 `j` 处的正向（输入）行波 $V^+_i$ 和反向（输出）行波 $V^-_j$，如下所示：$$V_i^+ = a_i \sqrt{\Re({Z_0}_i)} $$$$V_j^- = b_j \sqrt{\Re({Z_0}_j)} $$其中，${Z_0}_i$ 和 ${Z_0}_j$ 分别是端口 `i` 和 `j` 处的特性阻抗。

### 电压和行波在传输线理论中，总电压和总电流是入射电压波和反射电压波幅度的函数。类似地，可以通过使用在构成节点的元件端口上的正向（输入）和反向（输出）行波，来计算`Circuit`中每个节点的`Voltage`和`Current`。对于简单的串联连接，例如当`Network` $A$的端口`m`连接到`Network` $B$的端口`n`时，可以使用与传输线相同的原理来计算节点上的总`Voltage`：$$V=V_m^- + V_n^-$$这里，$V_m^-$和$V_n^-$分别是端口`m`和`n`上的反向（输出）行波。

### 阻抗不匹配时的电压当`Network` $A$的端口`m` (${Z_0}_{Am}$)和`Network` $B$的端口`n` (${Z_0}_{Bn}$)的端口特性阻抗不同时，会发生阻抗不匹配。 这需要使用传输系数$\Tau_{nm}$来校正不匹配。当电压波从`Network` $A$的端口`m`传输到`Network` $B$的端口`n`时，可以将`Network` $A$视为源，将`Network` $B`视为负载。然后可以使用从`m`到`n`的传输系数$\Tau_{nm}$来调整不匹配：$$ \Tau_{nm} = \frac{2 Z_{Load}}{Z_{Load}+Z_{Source}} = \frac{2 {Z_0}_{Am}}{{Z_0}_{Am}+{Z_0}_{Bn}}$$

### 具有并联连接的复杂电路在更复杂的`电路`中，例如具有并联连接的电路，多个端口可能会汇聚到一个节点，并且每个端口可能具有不同的特性阻抗。尽管增加了复杂性，但使用传输系数 $\Tau$ 的方法仍然有效。通过选择一个`网络`的特性阻抗 $Z_l$ 作为源阻抗，并将其他 $k$ 个并联`网络`的组合特性阻抗视为负载阻抗，可以确定端口 `l` 的实际传输系数：$$ \frac{1}{Z_{Source}}=\sum_{i=0}^{k}{\frac{1}{Z_{i}}}-\frac{1}{Z_l}$$这种方法允许计算节点处的有效阻抗，然后可以用于调整阻抗失配，并确保在复杂的并联`网络`配置中进行准确的`电压`和`电流`计算。

### 示例[威尔金森功率分配器](https://scikit-rf.readthedocs.io/en/latest/examples/circuit/Wilkinson%20Power%20Splitter.html) 提供了复杂电路中电压和电流计算的示例。它演示了使用分配器进行纯串联连接的 `Circuit` 的计算，以及涉及元件并联连接的更复杂的 `Circuit` 的计算，在两种情况下都产生一致的结果。